# TxData Exploration
Load TxData files, inspect them, and augment with disease/anatomy relationships from `data/disease_files`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "txdata_download.py").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from txdata_download import download_txdata_csvs, add_disease_anatomy_relationships


In [2]:
data_dir = repo_root / "data" / "txdata"
disease_files_dir = repo_root / "data" / "disease_files"
data_dir, disease_files_dir


(PosixPath('/Users/jkobject/Documents/code/TxGNN/data/txdata'),
 PosixPath('/Users/jkobject/Documents/code/TxGNN/data/disease_files'))

In [ ]:
# Optional: download raw TxData CSVs from source
# download_txdata_csvs(data_dir)


In [9]:
nodes = pd.read_csv(data_dir / "nodes.tab", sep="\t")
edges = pd.read_csv(data_dir / "edges.csv")
nodes.shape, edges.shape


((129375, 5), (8100498, 4))

In [ ]:
nodes.head()


In [ ]:
edges.relation.value_counts().head(20)


## Add Disease/Anatomy Relationships
This uses `data/disease_files/*.csv` and writes augmented files without overwriting originals.


In [4]:
new_nodes, new_edges, processed_files = add_disease_anatomy_relationships(
    nodes_path=data_dir / "nodes.tab",
    edges_path=data_dir / "edges.csv",
    disease_files_dir=disease_files_dir,
    output_nodes_path=data_dir / "nodes_augmented.tab",
    output_edges_path=data_dir / "edges_augmented.csv",
)
new_nodes, new_edges, processed_files


(380, 1607, 9)

In [7]:
new_edges

1607

In [8]:
nodes_aug = pd.read_csv(data_dir / "nodes_augmented.tab", sep="\t")
edges_aug = pd.read_csv(data_dir / "edges_augmented.csv")
nodes_aug.shape, edges_aug.shape


((129755, 5), (8102105, 4))

## now looking at OT

In [2]:
otdir = repo_root / "data" / "opentargets"

In [13]:
pd.read_parquet(otdir / "biosample")

,biosampleId,biosampleName,description,xrefs,synonyms,parents,ancestors,children,descendants
0,BFO_0000017,realizable entity,None,[],[],[BFO_0000020],"[BFO_0000001, BFO_0000020, BFO_0000002]","[BFO_0000016, BFO_0000023]","[BFO_0000016, BFO_0000034, BFO_0000023]"
1,BFO_0000050,part of,a core relation that holds between a part and ...,[BFO:0000050],[],None,[],None,[]
2,BSPO_0000054,anatomical side,An anatomical region bounded by a plane perpen...,[FBql:00005841],[],[BSPO_0000070],"[CARO_0000003, CARO_0000006, CARO_0000000, BSP...","[BSPO_0000000, BSPO_0000007]","[BSPO_0000000, BSPO_0000007]"
3,BSPO_0000111,right of,Closer to the right side of the organism. Exam...,[BSPO:0000111],[],None,[],None,[]
4,BSPO_0015014,http://purl.obolibrary.org/obo/BSPO_0015014,x immediately superficial to y iff x superfici...,[BSPO:0015014],[],None,[],None,[]
...,...,...,...,...,...,...,...,...,...
35686,uberon/core#anastomoses_with,anastomoses with,None,[],[],None,[],None,[]
35687,uberon/core#feed_aligned,http://purl.obolibrary.org/obo/uberon/core#fee...,None,[],[],None,[],None,[]
35688,uberon/core#inconsistent_with_fma,http://purl.obolibrary.org/obo/uberon/core#inc...,None,[],[],None,[],None,[]
35689,uberon/core#pheno_slim,http://purl.obolibrary.org/obo/uberon/core#phe...,None,[],[],None,[],None,[]


In [4]:
import pyarrow.parquet as pq


In [11]:
parquet_files = list((otdir / "credible_set").glob("*.parquet"))
pd.read_parquet(parquet_files[0])

,studyLocusId,studyId,variantId,chromosome,position,region,beta,zScore,pValueMantissa,pValueExponent,...,purityMeanR2,purityMinR2,locusStart,locusEnd,sampleSize,ldSet,locus,confidence,studyType,isTransQtl
0,88aab75d4dfa3d8a28a2de695f4743a3,onek1k_ge_cd4_tcm_ensg00000154328,8_11845988_G_T,8,11845988,chr8:10769639-12769639,-0.244950,NaN,2.325000,-9,...,NaN,NaN,NaN,NaN,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with in-sample LD,sceqtl,False
1,148f08f19b63731c83e409d9e7a61991,peng_2018_ge_placenta_naive_ensg00000164733,8_11845988_G_T,8,11845988,chr8:10869533-12869533,0.585279,NaN,3.045000,-12,...,NaN,NaN,NaN,NaN,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with in-sample LD,eqtl,False
2,e0be2f2a16ea6e77cf9f13fb5aa7ebf5,quach_2016_txrev_monocyte_pam3csk4_ensg0000016...,8_11846014_T_G,8,11846014,chr8:10869533-12869533,-0.783047,NaN,1.919000,-11,...,NaN,NaN,NaN,NaN,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with in-sample LD,tuqtl,False
3,4193ce3c7342abc75ac43e379200f8cf,gtex_txrev_esophagus_gej_ensg00000164733_grp_1...,8_11846014_T_G,8,11846014,chr8:10869533-12869533,0.376360,NaN,4.139000,-7,...,NaN,NaN,NaN,NaN,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with in-sample LD,tuqtl,False
4,6e2bd9d34fe87d61eae6795aca6df8f1,GCST90239670,8_11846014_T_G,8,11846014,8:11559085-11987782,0.007620,6.564260,5.229195,-11,...,0.967777,0.861400,11559085.0,11987782.0,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with out-of-sam...,gwas,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17655,80f19343454b84df6a3f447ef6e539a5,GCST90475724,8_20084640_T_TA,8,20084640,8:19536474-20323373,-0.012398,-13.093923,3.567000,-39,...,0.991652,0.981611,19536474.0,20323373.0,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with out-of-sam...,gwas,None
17656,9451cfbd1412a6a6e6496176cef28046,GCST90092890,8_20084640_T_TA,8,20084640,8:19548588-20222761,-0.021974,-12.651745,1.094172,-36,...,0.989261,0.970946,19548588.0,20222761.0,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with out-of-sam...,gwas,None
17657,5f11998db11f39aa0eb9af29a99a6a89,GCST90502555,8_20084640_T_TA,8,20084640,8:19548587-20299129,0.026401,14.559725,5.066097,-48,...,0.989259,0.970946,19548587.0,20299129.0,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with out-of-sam...,gwas,None
17658,8cf8fd612f0ca354f21743d9f1a0a243,GCST90269640,8_20084640_T_TA,8,20084640,8:19548588-20222335,-0.019711,-11.088484,1.426861,-28,...,0.970429,0.827502,19548588.0,20222335.0,NaN,None,"[{'is95CredibleSet': True, 'is99CredibleSet': ...",SuSiE fine-mapped credible set with out-of-sam...,gwas,None


In [19]:
parquet_files = list((otdir / "biosample").glob("*.parquet"))
pd.read_parquet(parquet_files[0])

,biosampleId,biosampleName,description,xrefs,synonyms,parents,ancestors,children,descendants
0,BFO_0000017,realizable entity,None,[],[],[BFO_0000020],"[BFO_0000001, BFO_0000020, BFO_0000002]","[BFO_0000016, BFO_0000023]","[BFO_0000016, BFO_0000034, BFO_0000023]"
1,BFO_0000050,part of,a core relation that holds between a part and ...,[BFO:0000050],[],None,[],None,[]
2,BSPO_0000054,anatomical side,An anatomical region bounded by a plane perpen...,[FBql:00005841],[],[BSPO_0000070],"[CARO_0000003, CARO_0000006, CARO_0000000, BSP...","[BSPO_0000000, BSPO_0000007]","[BSPO_0000000, BSPO_0000007]"
3,BSPO_0000111,right of,Closer to the right side of the organism. Exam...,[BSPO:0000111],[],None,[],None,[]
4,BSPO_0015014,http://purl.obolibrary.org/obo/BSPO_0015014,x immediately superficial to y iff x superfici...,[BSPO:0015014],[],None,[],None,[]
...,...,...,...,...,...,...,...,...,...
35686,uberon/core#anastomoses_with,anastomoses with,None,[],[],None,[],None,[]
35687,uberon/core#feed_aligned,http://purl.obolibrary.org/obo/uberon/core#fee...,None,[],[],None,[],None,[]
35688,uberon/core#inconsistent_with_fma,http://purl.obolibrary.org/obo/uberon/core#inc...,None,[],[],None,[],None,[]
35689,uberon/core#pheno_slim,http://purl.obolibrary.org/obo/uberon/core#phe...,None,[],[],None,[],None,[]


In [12]:
parquet_files = list((otdir / "target").glob("*.parquet"))
pd.read_parquet(parquet_files[0])

,id,approvedSymbol,biotype,transcriptIds,canonicalTranscript,canonicalExons,genomicLocation,alternativeGenes,approvedName,go,...,constraint,tep,proteinIds,dbXrefs,chemicalProbes,homologues,tractability,safetyLiabilities,pathways,tss
0,ENSG00000002587,HS3ST1,protein_coding,"[ENST00000952063, ENST00000952062, ENST0000000...","{'id': 'ENST00000002596', 'chromosome': '4', '...","[11428699, 11428894, 11393150, 11400113]","{'chromosome': '4', 'start': 11393150, 'end': ...",None,heparan sulfate-glucosamine 3-sulfotransferase 1,"[{'id': 'GO:0008467', 'source': 'GO_REF:000011...",...,"[{'constraintType': 'syn', 'score': -0.3221299...",None,"[{'id': 'O14792', 'source': 'uniprot_swissprot...","[{'id': '5194', 'source': 'HGNC'}, {'id': '1ZR...",None,"[{'speciesId': '9606', 'speciesName': 'Human',...","[{'modality': 'SM', 'id': 'Approved Drug', 'va...",None,"[{'pathwayId': 'R-HSA-2022928', 'pathway': 'HS...",11428894.0
1,ENSG00000002746,HECW1,protein_coding,"[ENST00000461842, ENST00000857209, ENST0000049...","{'id': 'ENST00000395891', 'chromosome': '7', '...","[43112647, 43112937, 43552222, 43552336, 43450...","{'chromosome': '7', 'start': 43112629, 'end': ...",None,"HECT, C2 and WW domain containing E3 ubiquitin...","[{'id': 'GO:0005829', 'source': 'Reactome:R-HS...",...,"[{'constraintType': 'syn', 'score': 0.00054383...",None,"[{'id': 'Q76N89', 'source': 'uniprot_swissprot...","[{'id': '22195', 'source': 'HGNC'}, {'id': '3L...",None,"[{'speciesId': '9606', 'speciesName': 'Human',...","[{'modality': 'SM', 'id': 'Approved Drug', 'va...",None,"[{'pathwayId': 'R-HSA-4641258', 'pathway': 'De...",43112647.0
2,ENSG00000002822,MAD1L1,protein_coding,"[ENST00000853801, ENST00000450235, ENST0000085...","{'id': 'ENST00000265854', 'chromosome': '7', '...","[2219332, 2219456, 1980453, 1980541, 2002065, ...","{'chromosome': '7', 'start': 1815787, 'end': 2...",None,mitotic arrest deficient 1 like 1,"[{'id': 'GO:0051315', 'source': 'GO_REF:000003...",...,"[{'constraintType': 'syn', 'score': -2.2871999...",None,"[{'id': 'Q9Y6D9', 'source': 'uniprot_swissprot...","[{'id': '6762', 'source': 'HGNC'}, {'id': '1GO...",None,"[{'speciesId': '9598', 'speciesName': 'Chimpan...","[{'modality': 'SM', 'id': 'Approved Drug', 'va...",None,"[{'pathwayId': 'R-HSA-2500257', 'pathway': 'Re...",2232945.0
3,ENSG00000002834,LASP1,protein_coding,"[ENST00000883575, ENST00000433206, ENST0000058...","{'id': 'ENST00000318008', 'chromosome': '17', ...","[38918605, 38921770, 38870058, 38870258, 38915...","{'chromosome': '17', 'start': 38869185, 'end':...",None,LIM and SH3 protein 1,"[{'id': 'GO:0005515', 'source': 'PMID:31515488...",...,"[{'constraintType': 'syn', 'score': 0.44841000...",None,"[{'id': 'Q14847', 'source': 'uniprot_swissprot...","[{'id': '6513', 'source': 'HGNC'}, {'id': '3I3...",None,"[{'speciesId': '9606', 'speciesName': 'Human',...","[{'modality': 'SM', 'id': 'Approved Drug', 'va...",None,None,38870058.0
4,ENSG00000002933,TMEM176A,protein_coding,"[ENST00000855178, ENST00000956681, ENST0000095...","{'id': 'ENST00000004103', 'chromosome': '7', '...","[150801536, 150801724, 150800769, 150800828, 1...","{'chromosome': '7', 'start': 150799598, 'end':...",None,transmembrane protein 176A,"[{'id': 'GO:0005515', 'source': 'PMID:32296183...",...,"[{'constraintType': 'syn', 'score': -0.3407100...",None,"[{'id': 'Q96HP8', 'source': 'uniprot_swissprot...","[{'id': '24930', 'source': 'HGNC'}, {'id': 'IP...",None,"[{'speciesId': '9606', 'speciesName': 'Human',...","[{'modality': 'SM', 'id': 'Approved Drug', 'va...",None,None,150800769.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7824,ENSG00000310503,ENSG00000310503,lncRNA,[ENST00000850487],"{'id': 'ENST00000850487', 'chromosome': '6', '...","[163945953, 163946062, 163958305, 163958692]","{'chromosome': '6', 'start': 163945953, 'end':...",None,novel transcript,None,...,None,None,None,[],None,None,None,None,None,163945953.0
7825,ENSG00000310510,ENSG00000310510,lncRNA,[ENST00000850542],"{